# 4. Fees & Yield — QA

**Purpose** — Validate fees, APR, yield curves, and the volume↔fees reconciliation. Canonical home for APR.

**Source tables**
- `mart_volume_daily`
- `mart_fee_yield_daily`  # deprecated view → mart_pool_screener_daily
- `int_fee_per_100_usd`
- `int_lp_macro_daily`
- `mart_price_analytics_daily`
- `dim_indexed_pools`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
fee_tier_pct = float(db.read_query(f"SELECT fee_tier FROM dim_indexed_pools WHERE lower(pool_address) = lower('{POOL}')")["fee_tier"][0])
fee_ratio = fee_tier_pct / 100.0  # 0.05 → 0.0005
print(f"Fee tier: {fee_tier_pct}% (ratio={fee_ratio})")

vol = db.read_query(f"""
SELECT date, volume_token0, volume_token1, volume_token0_equiv, swap_count, fees_token0, fees_token1, fees_token0_equiv
FROM mart_volume_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
fy = db.read_query(f"""
SELECT date, tvl_total_token0, tvl_active_token0, tvl_outofrange_token0, fees_token0_equiv, apr
FROM mart_fee_yield_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
fp100 = db.read_query(f"""
SELECT date, range_width_ticks, fee_per_100_token0_daily, fee_per_100_token0_7d, fee_per_100_token0_30d, fee_per_100_token0_90d,
       fee_bps_annual, active_liquidity, active_tvl_token0, close_tick, close_price
FROM int_fee_per_100_usd
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date, range_width_ticks
""")
print(f"vol:    {vol.height} rows ({vol['date'].min()} → {vol['date'].max()})")
print(f"fy:     {fy.height} rows")
print(f"fp100:  {fp100.height} rows  ({fp100['range_width_ticks'].unique().sort().to_list()} widths)")

Fee tier: 0.05000000074505806% (ratio=0.0005000000074505806)
vol:    1798 rows (2021-05-05 → 2026-04-06)
fy:     1798 rows
fp100:  10788 rows  ([50, 100, 200, 500, 1000, 2000] widths)


## §2 — Sanity checks

In [4]:
# Non-negativity
n_neg_fees   = vol.filter((pl.col("fees_token0") < 0) | (pl.col("fees_token0_equiv") < 0)).height
n_neg_volume = vol.filter(pl.col("volume_token0_equiv") < 0).height
qa_check("fees >= 0 in mart_volume_daily",  n_neg_fees == 0,   detail=f"violations={n_neg_fees}")
qa_check("volume >= 0 in mart_volume_daily", n_neg_volume == 0, detail=f"violations={n_neg_volume}")

# APR finiteness on mart_fee_yield_daily after warm-up (skip first 7d)
warm = fy.filter(pl.col("date") >= fy["date"].min() + timedelta(days=7))
n_apr_null = warm.filter(pl.col("apr").is_null() & pl.col("tvl_active_token0").gt(0)).height
qa_check("APR not null after warm-up (when active TVL > 0)",
         n_apr_null == 0, severity="warn", detail=f"days={n_apr_null}")

# fee_per_100 finiteness
n_fp_null = fp100.filter(
    pl.col("fee_per_100_token0_daily").is_null() & pl.col("active_liquidity").gt(0)
).height
qa_check("fee_per_100_daily not null when active_liquidity > 0",
         n_fp_null == 0, severity="warn", detail=f"violations={n_fp_null}")

🟢 PASS | fees >= 0 in mart_volume_daily  —  violations=0
🟢 PASS | volume >= 0 in mart_volume_daily  —  violations=0
🟢 PASS | APR not null after warm-up (when active TVL > 0)  —  days=0
🟢 PASS | fee_per_100_daily not null when active_liquidity > 0  —  violations=0


In [5]:
# Volume × fee_tier reconciliation: volume_token0 * ratio ≈ fees_token0 (input-side)
recon = db.read_query(f"""
SELECT
  count() AS rows,
  countIf(
    volume_token0_equiv > 0
    AND fees_token0_equiv > 0
    AND abs(volume_token0_equiv * {fee_ratio} - fees_token0_equiv) / fees_token0_equiv > 0.10
  ) AS mismatches_10pct
FROM mart_volume_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}') AND volume_token0_equiv > 0
""").row(0, named=True)
qa_check("fees ≈ volume_token0_equiv × fee_tier (within 10%)",
         int(recon["mismatches_10pct"]) <= 5, severity="warn",
         detail=f"days_off={recon['mismatches_10pct']} / {recon['rows']}")

🟢 PASS | fees ≈ volume_token0_equiv × fee_tier (within 10%)  —  days_off=0 / 1798


In [6]:
# APR-spike audit: a spike is "explained" if EITHER
#   (a) active TVL is in the bottom decile (low denominator) OR
#   (b) realized_vol_7d is in the top decile (genuine high-fee regime)
spike_q = f"""
WITH apr AS (
  SELECT fy.date, fy.apr, fy.tvl_active_token0,
         median(fy.apr) OVER (PARTITION BY fy.pool ORDER BY fy.date ROWS BETWEEN 30 PRECEDING AND CURRENT ROW) AS med30,
         quantileExact(0.10)(fy.tvl_active_token0) OVER (PARTITION BY fy.pool ORDER BY fy.date ROWS BETWEEN 60 PRECEDING AND CURRENT ROW) AS q10_tvl,
         pa.realized_vol_7d AS rv,
         quantileExact(0.90)(pa.realized_vol_7d) OVER (PARTITION BY fy.pool ORDER BY fy.date ROWS BETWEEN 60 PRECEDING AND CURRENT ROW) AS q90_rv
  FROM mart_fee_yield_daily fy
  INNER JOIN mart_price_analytics_daily pa USING (pool, date)
  WHERE lower(fy.pool) = lower('{POOL}') AND fy.date <= toDate('{END_DATE}')
)
SELECT
  countIf(apr > 5 * med30 AND tvl_active_token0 > q10_tvl AND rv < q90_rv) AS unexplained_spikes,
  countIf(apr > 5 * med30) AS total_spikes
FROM apr
"""
spike = db.read_query(spike_q).row(0, named=True)
qa_check("APR spikes (>5× rolling-30d median) explained by low TVL OR high realized vol",
         int(spike["unexplained_spikes"]) <= 2, severity="warn",
         detail=f"unexplained={spike['unexplained_spikes']}/{spike['total_spikes']}")


🟢 PASS | APR spikes (>5× rolling-30d median) explained by low TVL OR high realized vol  —  unexplained=1/32


## §3 — Diagnostic visuals

**QA purpose**: confirm fees ≈ volume × fee_tier. Residuals near zero means swap-by-swap fee math is consistent with daily aggregation.

In [7]:
recon_df = vol.with_columns(
    (pl.col("volume_token0_equiv") * fee_ratio).alias("est_fees"),
    pl.col("fees_token0_equiv").alias("rep_fees"),
).with_columns(
    (pl.col("rep_fees") - pl.col("est_fees")).alias("residual"),
).filter(pl.col("rep_fees") > 0).filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Estimated vs reported daily fees", "Residual = reported - estimated"))
fig.add_trace(go.Scatter(x=recon_df["date"].to_list(), y=recon_df["est_fees"].to_list(), name="est fees", line=dict(color="#3b82f6")), row=1, col=1)
fig.add_trace(go.Scatter(x=recon_df["date"].to_list(), y=recon_df["rep_fees"].to_list(), name="reported fees", line=dict(color="#22c55e")), row=1, col=1)
fig.add_trace(go.Bar(x=recon_df["date"].to_list(), y=recon_df["residual"].to_list(), name="residual"), row=2, col=1)
fig.update_layout(height=480, title_text="Fee reconciliation (last 90d)")
fig.show()

**QA purpose**: APR daily with TVL on secondary axis — spikes annotated with TVL helps audit visually.

In [8]:
fy_90 = fy.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=fy_90["date"].to_list(), y=fy_90["apr"].to_list(),
                          name="apr_daily", line=dict(color="#22c55e")), secondary_y=False)
fig.add_trace(go.Scatter(x=fy_90["date"].to_list(), y=fy_90["tvl_active_token0"].to_list(),
                          name="active TVL", line=dict(color="#9ca3af", dash="dot")), secondary_y=True)
fig.update_layout(height=360, title_text="APR daily vs active TVL (last 90d)")
fig.show()

**QA purpose**: yield curve smoothness across widths and horizons. Curves should be smooth and monotonically related across horizons (1d, 7d, 30d, 90d).

In [9]:
latest_d = fp100["date"].max()
last_curve = fp100.filter(pl.col("date") == latest_d).sort("range_width_ticks")
fig = go.Figure()
for col, label in [("fee_per_100_token0_daily", "1d"), ("fee_per_100_token0_7d", "7d"), ("fee_per_100_token0_30d", "30d"), ("fee_per_100_token0_90d", "90d")]:
    fig.add_trace(go.Scatter(x=last_curve["range_width_ticks"].to_list(), y=last_curve[col].to_list(), name=label, mode="lines+markers"))
fig.update_layout(height=360, title_text=f"Fee-per-100-token0 yield curves — {latest_d}", xaxis_title="range_width_ticks")
fig.show()

**QA purpose**: APR vs realized vol scatter — sanity that fee yield correlates with vol over the historical window.

In [10]:
prc = db.read_query(f"""
SELECT date, realized_vol_7d FROM mart_price_analytics_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""")
joined = fy.join(prc, on="date").filter(pl.col("apr").is_not_null() & pl.col("realized_vol_7d").is_not_null())
fig = go.Figure(go.Scatter(
    x=joined["realized_vol_7d"].to_list(),
    y=joined["apr"].to_list(),
    mode="markers", marker=dict(size=4, opacity=0.4, color=joined["tvl_active_token0"].to_list(), colorscale="Viridis", showscale=True, colorbar_title="active TVL"),
    text=[str(d) for d in joined["date"].to_list()],
    hovertemplate="%{text}<br>σ7d=%{x:.3f}<br>apr=%{y:.3f}<extra></extra>",
))
fig.update_layout(height=420, title_text="APR daily vs realized vol 7d", xaxis_title="realized_vol_7d", yaxis_title="apr_daily")
fig.show()

## §4 — QA verdict

In [11]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 6   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""fees >= 0 in mart_volume_daily""","""PASS""","""fail""","""violations=0"""
"""volume >= 0 in mart_volume_dai…","""PASS""","""fail""","""violations=0"""
"""APR not null after warm-up (wh…","""PASS""","""warn""","""days=0"""
"""fee_per_100_daily not null whe…","""PASS""","""warn""","""violations=0"""
"""fees ≈ volume_token0_equiv × f…","""PASS""","""warn""","""days_off=0 / 1798"""
"""APR spikes (>5× rolling-30d me…","""PASS""","""warn""","""unexplained=1/32"""
